# SOTA verification - Mip-NeRF 360 (`garden`)

The trust-builder: train static **`premium`** with foundation depth + NVS eval on a
published benchmark scene, then compare to the baseline with `scripts/sota_compare.py`.
Turns "looks good on my room" into a baseline-anchored verdict.

Runtime: **A100 GPU**. ~2-6 h total: CPU COLMAP exhaustive matching is the slow step
(~1-3 h for ~185 images), then Metric3D depth + 100k-iter training + eval.


## 0. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 1. Clone repo (branch `chore/strip-to-core`)

In [ ]:
import os
REPO_URL = "https://github.com/mehmettahacumurcu/gaussian-splatter.git"
BRANCH   = "chore/strip-to-core"   # Phase 2 work lives here, NOT main
if not os.path.exists("gaussian-splatter"):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL}
%cd gaussian-splatter
!git log --oneline -1

## 2. Environment (deps + gsplat + COLMAP)

In [ ]:
!bash colab/bootstrap.sh --colmap

In [ ]:
# If this errors about numpy: Runtime -> Restart session, then re-run THIS cell only.
# (The chdir guard below re-enters the repo after a restart resets cwd to /content.)
import os
if os.path.isdir("/content/gaussian-splatter"):
    os.chdir("/content/gaussian-splatter")
import torch, gsplat
print("torch", torch.__version__, "| gsplat", gsplat.__version__,
      "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 3. Download a Mip-NeRF 360 scene

`garden` (outdoor) baseline = 27.41 dB / vanilla-3DGS - the fairest comparison for our
pipeline family. Swap to `bonsai` (indoor, 32.70 dB / Scaffold-GS) by editing `SCENE`.
We use the 1/4-res `images_4` (the 3DGS outdoor eval standard) when available.


In [ ]:
SCENE = "garden"   # alt: "bonsai", "room", "counter", "kitchen", "stump", "bicycle"
import os, glob, shutil, subprocess
os.makedirs("data", exist_ok=True)
dst_imgs = f"data/{SCENE}/images"
if not glob.glob(dst_imgs + "/*"):
    if not os.path.exists("/content/360_v2.zip"):
        subprocess.run(["wget", "-q", "-O", "/content/360_v2.zip",
                        "http://storage.googleapis.com/gresearch/refraw360/360_v2.zip"], check=True)
    subprocess.run(["unzip", "-q", "-o", "/content/360_v2.zip", f"{SCENE}/*", "-d", "/content/m360"], check=True)
    src = f"/content/m360/{SCENE}/images_4"           # 3DGS outdoor eval res (~1.3k px)
    if not os.path.isdir(src):
        src = f"/content/m360/{SCENE}/images"
    os.makedirs(f"data/{SCENE}", exist_ok=True)
    shutil.copytree(src, dst_imgs)
    os.remove("/content/360_v2.zip")             # free ~11 GB (re-downloads if you switch SCENE)
    shutil.rmtree(f"/content/m360/{SCENE}")      # full-res copies not needed; we train on images_4
print(SCENE, "images:", len(glob.glob(dst_imgs + "/*")))

## 4. Train static `premium` + foundation + NVS eval (the long cell)

Runs COLMAP (pose estimation) -> Metric3D depth -> training -> every-8 held-out eval,
end-to-end. This is the honest test of the full pipeline, not a shortcut.

Two flags matter here:
- **`--colmap-cpu`** — the apt COLMAP build has no CUDA and GPU SIFT needs a display, so
  GPU SIFT crashes on headless Colab. CPU SIFT is the deterministic path; exhaustive
  matching of ~185 images is the slow step (roughly 1-3 h).
- **`--native-res`** — train/eval at the source `images_4` resolution (~1297x840) instead
  of the preset's fixed 2560x1440. The published 27.41 dB baseline is measured at native
  `images_4` resolution; without this flag the eval upsamples + aspect-stretches GT and
  the verdict is not comparable.

In [ ]:
import time
RUN_START = time.time()   # freshness anchor: the verdict cell refuses an eval file older than this
!python scripts/static_3dgs.py --scene {SCENE} --preset premium --foundation --nvs-eval --colmap-cpu --native-res

## 5. Verdict vs published baseline

In [ ]:
import sys
sys.path.insert(0, ".")
from colab.verify_helpers import assert_fresh_eval
# A re-run that crashed before Faz 7 leaves the PREVIOUS nvs_eval.json in place,
# and sota_compare would happily grade the stale file. Fail loudly instead.
assert_fresh_eval(SCENE, RUN_START)
!python scripts/sota_compare.py {SCENE}

**Reading it:**
- `SOTA-tier` (delta >= -1.0 dB) -> algorithm is sound; compute (cloud) is the next lever.
- `Below SOTA` (-3.0 .. -1.0) -> tunable; try a stronger preset / check foundation + density.
- `Algorithmic gap` (< -3.0) -> compute won't fix it; check eval protocol, COLMAP, foundation.


## 6. Save to Drive - splat (always) + results + walkable world

The splat `.ply` is copied first and unconditionally, so you have it even if the world-wrap step errors.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, ".")
from colab.verify_helpers import save_splat_to_drive, copy_results_to_drive, wrap_and_save_world

# (a) raw splat .ply - saved FIRST, always (open in any 3DGS viewer)
save_splat_to_drive(SCENE, "/content/drive/MyDrive/4dgs")
# (b) full results (eval json, logs, orbit.mp4)
copy_results_to_drive(SCENE, "/content/drive/MyDrive/4dgs/results")
# (c) walkable world bundle - best effort; the splat above is safe even if this fails
try:
    wrap_and_save_world(SCENE, "/content/drive/MyDrive/4dgs")
except Exception as e:
    print("  world wrap skipped:", e, "\n  -> the splat .ply above is still on Drive")

**View it.**
- *Splat only* (simplest): open `MyDrive/4dgs/splats/garden.ply` in any 3DGS viewer, e.g.
  https://superspl.at/editor - no project needed.
- *Walkable*: download `MyDrive/4dgs/worlds/garden/` into your local `worlds/garden/`,
  run `cd frontend && npm run dev`, open the **Interactive** page, pick **"Garden (Mip-NeRF360, Colab SOTA)"** in the
  world dropdown (top-right).